In [34]:
import cv2
import mediapipe as mp
import os
from google.protobuf.json_format import MessageToDict
# import json
import numpy as np
from io import BytesIO
from PIL import Image


import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from model_dataset import FixedHandshapeDict, GuideReader
from paths import *
from model_model import LinearHandshapePredictor
from model_configs import *
from utils import *
from recorder import *
from graph_tools import GraphTool, Plotter, Smoother

In [17]:
FLAG_NONE = 0
FLAG_OK = 1
FLAG_FILLED = 2 # set FLAG_FILLED = 1 if include interpolated data in interpolation next round

def lm_has_side_and_is_at(lm, side):
    # This is a renewed version of lm_has_side_and_is_at, considering the order is not strict
    # Use "Left", and "Right" for side, instead of numbers
    mh = lm.multi_handedness
    if mh is None: 
        return False, 0

    max_score = -1
    max_index = 0
    found = False
    for i, hand in enumerate(mh):
        handedness_dict = MessageToDict(hand)
        classification = handedness_dict["classification"][0]
        if classification["label"] == side:
            if classification["score"] > max_score:
                max_score = classification["score"]
                max_index = i
                found = True
    return found, max_index

def sol2json(d, json_path, side): 
    with open(json_path, 'w') as fl:
        has, at = lm_has_side_and_is_at(d, side)
        if has: 
            ml = (MessageToDict(d.multi_hand_landmarks[at])["landmark"]) # 0 is one of the hands, do this first
            my_dict = {str(i): (d['x'], d['y'], d['z']) for i, d in enumerate(ml)}
            this_flag = FLAG_OK
        else: 
            my_dict = {str(i): (0, 0, 0) for i in range(21)} # default (0, 0, 0) for all nodes
            this_flag = FLAG_NONE
        # outdict = {"edges": el, "features": my_dict}  # for the current processing, it is not needed to include edges
        outdict = {"features": my_dict, "flag": this_flag}
        # fl.write(json.dumps(outdict, separators=(',', ':')))
        return outdict
    
def dict2array(dict_data):
    feature_list = [dict_data['features'][str(i)] for i in range(21)]
    feature_array = np.array(feature_list)
    return feature_array.reshape(1, 21, 3)

## Load Model

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()
model = LinearHandshapePredictor(
    input_dim=in_dim, 
    enc_lat_dims=enc_lat_dims, 
    hid_dim=hid_dim, 
    dec_lat_dims=dec_lat_dims, 
    output_dim=out_dim
)
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

ts = "1113174414-lin"
stop_epoch = "597"
save_subdir = os.path.join(model_save_dir, "{}/".format(ts))
model_raw_name = f"{stop_epoch}"
model_name = model_raw_name + ".pt"
model_path = os.path.join(save_subdir, model_name)
state = torch.load(model_path)
model.load_state_dict(state)
model.to(device)
model.eval()

LinearHandshapePredictor(
  (encoder): Sequential(
    (0): LinPack(
      (lin): Linear(in_features=63, out_features=128, bias=True)
      (relu): ReLU()
      (batch_norm): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResBlock(
      (lin1): Linear(in_features=128, out_features=128, bias=True)
      (lin2): Linear(in_features=128, out_features=128, bias=True)
      (relu): ReLU()
    )
    (2): LinPack(
      (lin): Linear(in_features=128, out_features=32, bias=True)
      (relu): ReLU()
      (batch_norm): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (3): ResBlock(
      (lin1): Linear(in_features=32, out_features=32, bias=True)
      (lin2): Linear(in_features=32, out_features=32, bias=True)
      (relu): ReLU()
    )
    (4): Linear(in_features=32, out_features=5, bias=True)
    (5): Sigmoid()
  )
  (decoder): Sequential(
    (0): LinPack(
      (lin): Linear(in_features=5, out_features

In [19]:
hsdict = FixedHandshapeDict()
def model_predict(model, features, hsdict): 
    this_features = torch.from_numpy(features)
    batch_num, lm_num, dim_num = this_features.size()
    x = this_features
    x = x.to(device)
    x = x.to(torch.float32)
    hid_rep, pred = model.predict(x, hsdict)
    hid_rep = hid_rep.cpu().detach().numpy()
    return hid_rep, pred

In [41]:
class ListBuffer: 
    def __init__(self, buffer_size) -> None:
        self.buffer = []
        self.buffer_size = buffer_size
    def append(self, item): 
        self.buffer.append(item)
        if len(self.buffer) > self.buffer_size: 
            self.buffer.pop(0)
    def get(self): 
        return self.buffer
    def stack_and_get(self): 
        return np.stack(self.buffer, axis=0)

In [50]:
class HandDetection:
    def __init__(self, video_path=None):
        self.mp_hands = mp.solutions.hands
        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles
        self.handshape_dir = "./hs_char/"
        self.saved_frames_hidrep_right = ListBuffer(10)
        self.saved_frames_hidrep_left = ListBuffer(10)

        # Try opening the webcam (index 0). If unavailable, load a video file.
        if video_path and os.path.exists(video_path):
            print(f"Using local video: {video_path}")
            self.cap = cv2.VideoCapture(video_path)
        else:
            print("Trying to use the webcam...")
            self.cap = cv2.VideoCapture(0)
            if not self.cap.isOpened():
                print("No webcam found! Please provide a valid video file.")
                exit(1)

    def overlay_handshape(self, frame, prediction, position=(50, 50), size=(100, 100)):
        """Overlay predicted handshape image on the frame with a white background."""
        handshape_path = os.path.join(self.handshape_dir, f"{prediction}.png")
        if os.path.exists(handshape_path):
            handshape_img = cv2.imread(handshape_path, cv2.IMREAD_UNCHANGED)
            handshape_img = cv2.resize(handshape_img, size)

            # Ensure the image has an alpha channel (RGBA)
            if handshape_img.shape[2] == 4:
                overlay = handshape_img[:, :, :3]  # Extract RGB
                alpha = handshape_img[:, :, 3] / 255.0  # Normalize alpha
            else:
                overlay = handshape_img
                alpha = np.ones((size[1], size[0]))  # No transparency

            # Create a white background of the same size
            white_bg = np.ones((size[1], size[0], 3), dtype=np.uint8) * 255

            # Blend handshape image with the white background
            blended_img = (1 - alpha[:, :, None]) * white_bg + alpha[:, :, None] * overlay

            # Overlay the final image on the frame
            x, y = position
            h, w = blended_img.shape[:2]
            frame[y:y+h, x:x+w] = blended_img

        return frame

    def plot_hidden_representation(self, hidden_rep, size=(200, 100)):
        """Generate and return a small line plot of the hidden representation.
        
        - Each dimension is plotted as a different colored line.
        - The x-axis represents time (frames).
        - A white background is used for better visibility.
        """
        fig, ax = plt.subplots(figsize=(2, 1), dpi=100)

        # Plot each dimension as a different colored line
        num_dims = hidden_rep.shape[1]  # Assuming hidden_rep is (frames, dimensions)
        for dim in range(num_dims):
            ax.plot(hidden_rep[:, dim], linewidth=1.5, label=f'Dim {dim+1}')

        # Set the x-axis as time (frames)
        ax.set_xlabel("Time (Frames)", fontsize=6)
        ax.set_ylabel("Hidden Representation", fontsize=6)

        # Style adjustments
        ax.set_xticks([])
        ax.set_yticks([])
        ax.legend(fontsize=4, loc='upper right', frameon=False)  # Small legend
        ax.set_facecolor("white")  # Ensure white background
        fig.patch.set_facecolor("white")

        # Convert plot to image
        buf = BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0, dpi=100)
        plt.close(fig)  # Close the figure to free memory

        buf.seek(0)
        img = Image.open(buf).convert("RGB")
        img = img.resize(size)

        return np.array(img)


    def detect(self):
        with self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5) as hands:
            counter = 0
            
            while self.cap.isOpened():
                ret, frame = self.cap.read()
                if not ret:
                    print("End of video or camera disconnected.")
                    break

                counter += 1

                # Convert the BGR frame to RGB and flip for correct handedness output
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                rgb_frame = cv2.flip(rgb_frame, 1)  # Mirror effect
                
                # Process the frame
                results = hands.process(rgb_frame)
                right_lm = sol2json(results, os.path.join("jsonsave", f"hand_landmarks_{counter}.json"), "Right")
                left_lm = sol2json(results, os.path.join("jsonsave", f"hand_landmarks_{counter}.json"), "Left")

                right_lm_array = dict2array(right_lm)
                left_lm_array = dict2array(left_lm)
                # print(right_lm_array.shape)
                hid_rep_right, pred_right = model_predict(model, right_lm_array, hsdict)
                hid_rep_left, pred_left = model_predict(model, left_lm_array, hsdict)
                self.saved_frames_hidrep_right.append(hid_rep_right[0])
                self.saved_frames_hidrep_left.append(hid_rep_left[0])

                annotated_frame = cv2.flip(frame.copy(), 1)
                # Draw hand landmarks
                if results.multi_hand_landmarks:
                    for hand_landmarks in results.multi_hand_landmarks:
                        self.mp_drawing.draw_landmarks(
                            annotated_frame,  # Draw on original BGR frame
                            hand_landmarks,
                            self.mp_hands.HAND_CONNECTIONS,
                            self.mp_drawing_styles.get_default_hand_landmarks_style(),
                            self.mp_drawing_styles.get_default_hand_connections_style()
                        )
                annotated_frame = self.overlay_handshape(annotated_frame, pred_right[0], position=(50, 50))
                annotated_frame = self.overlay_handshape(annotated_frame, pred_left[0], position=(50, 200))
                # print(self.saved_frames_hidrep_right.stack_and_get().shape)
                # raise Exception
                hidden_plot_size = (400, 200)
                hidden_plot_right = self.plot_hidden_representation(self.saved_frames_hidrep_right.stack_and_get(), size=hidden_plot_size)
                hidden_plot_left = self.plot_hidden_representation(self.saved_frames_hidrep_left.stack_and_get(), size=hidden_plot_size)
                annotated_frame[50:250, 300:700] = cv2.resize(hidden_plot_right, hidden_plot_size)  # Overlay plot
                annotated_frame[250:450, 300:700] = cv2.resize(hidden_plot_left, hidden_plot_size)  # Overlay plot
                # Display the frame
                cv2.imshow("Hand Detection", annotated_frame)

                # Press 'q' to exit the loop
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

        # Release resources
        self.cap.release()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    # video_path = "./B_01_011-START-15CB-2.mp4"  # Change to your local video file or leave None for webcam
    video_path = "./B_01_079-NOTHING-0KC7-67.mp4"
    detector = HandDetection(video_path=video_path)
    detector.detect()


Using local video: ./B_01_079-NOTHING-0KC7-67.mp4


QObject::moveToThread: Current thread (0x6db20060) is not the object's thread (0x6db3b7c0).
Cannot move to target thread (0x6db20060)

QObject::moveToThread: Current thread (0x6db20060) is not the object's thread (0x6db3b7c0).
Cannot move to target thread (0x6db20060)

QObject::moveToThread: Current thread (0x6db20060) is not the object's thread (0x6db3b7c0).
Cannot move to target thread (0x6db20060)

QObject::moveToThread: Current thread (0x6db20060) is not the object's thread (0x6db3b7c0).
Cannot move to target thread (0x6db20060)

QObject::moveToThread: Current thread (0x6db20060) is not the object's thread (0x6db3b7c0).
Cannot move to target thread (0x6db20060)

QObject::moveToThread: Current thread (0x6db20060) is not the object's thread (0x6db3b7c0).
Cannot move to target thread (0x6db20060)

QObject::moveToThread: Current thread (0x6db20060) is not the object's thread (0x6db3b7c0).
Cannot move to target thread (0x6db20060)

QObject::moveToThread: Current thread (0x6db20060) is n

End of video or camera disconnected.


In [ ]:
def overlay_handshape(self, frame, prediction, position=(50, 50), size=(100, 100)):
    """Overlay predicted handshape image on the frame with a white background."""
    handshape_path = os.path.join(self.handshape_dir, f"{prediction}.png")
    if os.path.exists(handshape_path):
        handshape_img = cv2.imread(handshape_path, cv2.IMREAD_UNCHANGED)
        handshape_img = cv2.resize(handshape_img, size)

        # Ensure the image has an alpha channel (RGBA)
        if handshape_img.shape[2] == 4:
            overlay = handshape_img[:, :, :3]  # Extract RGB
            alpha = handshape_img[:, :, 3] / 255.0  # Normalize alpha
        else:
            overlay = handshape_img
            alpha = np.ones((size[1], size[0]))  # No transparency

        # Create a white background of the same size
        white_bg = np.ones((size[1], size[0], 3), dtype=np.uint8) * 255

        # Blend handshape image with the white background
        blended_img = (1 - alpha[:, :, None]) * white_bg + alpha[:, :, None] * overlay

        # Overlay the final image on the frame
        x, y = position
        h, w = blended_img.shape[:2]
        frame[y:y+h, x:x+w] = blended_img

    return frame


In [13]:
import cv2
import mediapipe as mp
import numpy as np
import os
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from model import model_predict  # Assuming your model prediction function is in model.py

class HandDetectionWithOverlay:
    def __init__(self, model, hsdict, video_path=None, handshape_dir="handshapes/"):
        self.model = model
        self.hsdict = hsdict
        self.handshape_dir = handshape_dir

        self.mp_hands = mp.solutions.hands
        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles

        # Try webcam or load video file
        if video_path and os.path.exists(video_path):
            print(f"Using local video: {video_path}")
            self.cap = cv2.VideoCapture(video_path)
        else:
            print("Trying to use the webcam...")
            self.cap = cv2.VideoCapture(0)
            if not self.cap.isOpened():
                print("No webcam found! Please provide a valid video file.")
                exit(1)

        # Get frame size and FPS for video saving
        self.frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)

        # Define output video writer
        self.out_vid = cv2.VideoWriter(
            "output_with_overlay.mp4",
            cv2.VideoWriter_fourcc(*"mp4v"),
            self.fps,
            (self.frame_width, self.frame_height)
        )

    def overlay_handshape(self, frame, prediction, position=(50, 50), size=(100, 100)):
        """Overlay predicted handshape image on the frame."""
        handshape_path = os.path.join(self.handshape_dir, f"{prediction}.png")
        if os.path.exists(handshape_path):
            handshape_img = cv2.imread(handshape_path, cv2.IMREAD_UNCHANGED)
            handshape_img = cv2.resize(handshape_img, size)

            # Convert handshape image to RGBA if it has an alpha channel
            if handshape_img.shape[2] == 4:
                overlay = handshape_img[:, :, :3]  # Extract RGB
                alpha = handshape_img[:, :, 3] / 255.0  # Normalize alpha
            else:
                overlay = handshape_img
                alpha = np.ones((size[1], size[0]))  # No transparency

            # Overlay the handshape image on the frame
            x, y = position
            h, w = overlay.shape[:2]
            frame[y:y+h, x:x+w] = (1 - alpha[:, :, None]) * frame[y:y+h, x:x+w] + alpha[:, :, None] * overlay

        return frame

    def plot_hidden_representation(self, hidden_rep, position=(300, 50), size=(200, 100)):
        """Generate and return a small line plot of the hidden representation."""
        fig, ax = plt.subplots(figsize=(2, 1))
        ax.plot(hidden_rep, color='blue', linewidth=2)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)

        # Convert plot to image
        buf = BytesIO()
        plt.savefig(buf, format="png", transparent=True, bbox_inches="tight", pad_inches=0)
        buf.seek(0)
        img = Image.open(buf).convert("RGB")
        img = img.resize(size)

        return np.array(img)

    def detect(self):
        with self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5) as hands:
            
            while self.cap.isOpened():
                ret, frame = self.cap.read()
                if not ret:
                    print("End of video or camera disconnected.")
                    break

                # Convert BGR frame to RGB
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                rgb_frame = cv2.flip(rgb_frame, 1)  # Mirror effect

                # Process hand landmarks
                results = hands.process(rgb_frame)

                # Prepare for model input
                right_lm_array = None
                left_lm_array = None

                if results.multi_hand_landmarks:
                    for idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
                        # Draw landmarks
                        self.mp_drawing.draw_landmarks(
                            frame,
                            hand_landmarks,
                            self.mp_hands.HAND_CONNECTIONS,
                            self.mp_drawing_styles.get_default_hand_landmarks_style(),
                            self.mp_drawing_styles.get_default_hand_connections_style()
                        )

                        # Collect landmarks
                        lm_array = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark])
                        if idx == 0:
                            right_lm_array = lm_array
                        else:
                            left_lm_array = lm_array

                # Get model predictions
                if right_lm_array is not None:
                    hid_rep_right, pred_right = model_predict(self.model, right_lm_array, self.hsdict)
                    frame = self.overlay_handshape(frame, pred_right, position=(50, 50))
                    hidden_plot = self.plot_hidden_representation(hid_rep_right)
                    frame[50:150, 300:500] = cv2.resize(hidden_plot, (200, 100))  # Overlay plot

                if left_lm_array is not None:
                    hid_rep_left, pred_left = model_predict(self.model, left_lm_array, self.hsdict)
                    frame = self.overlay_handshape(frame, pred_left, position=(50, 200))
                    hidden_plot = self.plot_hidden_representation(hid_rep_left)
                    frame[200:300, 300:500] = cv2.resize(hidden_plot, (200, 100))  # Overlay plot

                # Show and save the frame
                cv2.imshow("Hand Detection with Overlays", frame)
                self.out_vid.write(frame)

                # Press 'q' to exit
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

        # Release resources
        self.cap.release()
        self.out_vid.release()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    video_path = "path_to_local_video.mp4"  # Set this to None for webcam
    model = load_model("path_to_model.pth")  # Load your trained model
    hsdict = load_hsdict("path_to_handshape_dict.pkl")  # Load handshape dictionary

    detector = HandDetectionWithOverlay(model, hsdict, video_path=video_path)
    detector.detect()


NameError: name 'results' is not defined